# Guía Olist


In [1]:
import pandas as pd

orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
order_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
geo = pd.read_csv("olist_geolocation_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")

In [2]:
a = payments["payment_type"]=="not_defined"
a.sum()

np.int64(3)

In [3]:
# de la tabla pedidos muestra únicamente order_id, customer_id y order_status
order_items.columns 

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')

In [4]:
sellers.columns

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')

In [5]:
# 4. Muestra los artículos cuyo precio sea mayor que 100.
#order_items[order_items["price"]>100]
pr_it = products.merge(order_items, on="product_id", how = "left")
pr_it[pr_it["price"]>100][["product_id", "price"]]

,product_id,price
1,3aa071139cb16b67ca9e5dea641aaa2f,248.0
3,cef67bcfe19066a932b7673e239eb23d,112.3
6,732bd381ad09e530fe0a5f457d81becb,958.0
7,732bd381ad09e530fe0a5f457d81becb,968.0
23,03b63c5fc16691530586ae020c345514,239.0
...,...,...
112638,9a7c6041fa9592d9d9ef6cfe62a71f8c,127.5
112639,9a7c6041fa9592d9d9ef6cfe62a71f8c,127.5
112640,9a7c6041fa9592d9d9ef6cfe62a71f8c,127.5
112641,9a7c6041fa9592d9d9ef6cfe62a71f8c,127.0


In [6]:
# 17 reemplazar los vacíos product_weight por la mediana
mediana_peso = products["product_weight_g"].median()
products["product_weight_g"] = products["product_weight_g"].fillna(mediana_peso)
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                0
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [7]:
#12 hallar productos que en su categoria tengan la palabra "beleza"
products["product_category_name"] = products["product_category_name"].fillna("unkwnown")
filtro_beleza = products["product_category_name"].str.contains("beleza")
products[filtro_beleza]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
22,e3e020af31d4d89d2602272b315c3f6e,beleza_saude,58.0,1987.0,3.0,75.0,21.0,7.0,13.0
49,c5d8079278e912d7e3b6beb48ecb56e8,beleza_saude,60.0,330.0,1.0,1300.0,19.0,28.0,17.0
62,36555a2f528d7b2a255c504191445d39,beleza_saude,51.0,1069.0,1.0,1467.0,23.0,16.0,20.0
75,e586ebb6022265ae1eea38f46ffe3ead,beleza_saude,40.0,1469.0,3.0,1050.0,18.0,21.0,27.0
80,75b4372e69a42f8ae1d908c076f547b2,beleza_saude,22.0,1313.0,1.0,938.0,29.0,22.0,23.0
...,...,...,...,...,...,...,...,...,...
32895,ff0ba1ffc67724ddc6db331bd417ef67,beleza_saude,49.0,1029.0,1.0,517.0,24.0,14.0,14.0
32915,d790fcd60d4743d309fedf7e7bb4c902,beleza_saude,39.0,811.0,1.0,650.0,50.0,5.0,30.0
32917,4419a003d42a243227478f2a949722d8,beleza_saude,27.0,1026.0,2.0,850.0,25.0,6.0,21.0
32918,2636a9812d151c718b53387171ce08be,beleza_saude,52.0,2088.0,1.0,443.0,21.0,13.0,17.0


In [8]:
# Selecciona los vendedores del estado SP, mostrando identificador, ciudad y estado.
sellers[sellers["seller_state"] == "SP"][["seller_id", "seller_city", "seller_state"]]

,seller_id,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,mogi guacu,SP
3,c0f3eea2e14555b6faeea3dd58c1b1c3,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,braganca paulista,SP
7,1b938a7ec6ac5061a66a3766e0e75f90,penapolis,SP
...,...,...,...
3087,f00f5b35d0abcacbdd863672f4bb2c1a,sao paulo,SP
3088,7bac63f6603d382cc8d0832eb6c100a8,sao paulo,SP
3089,f1fdf2d13186575751aa25876536d85c,sao paulo,SP
3092,74871d19219c7d518d0090283e03c137,sao paulo,SP


In [9]:
#Para cada vendedor, calcula número de pedidos distintos, número de artículos, precio total y
#flete total. Evita contar varias veces un pedido dentro de cada vendedor
df_46 = order_items.merge(orders, on = "order_id", how ="left")
df_46.groupby("seller_id").agg(
    ventas_unicas =("order_id", "nunique"),
    articulos_vendidos = ("order_id", "count"),
    productos_distinitos = ("product_id", "nunique"),
    total_flete =("freight_value", "sum"),
    precio_total = ("price" , "sum")
).sort_values(by="precio_total", ascending=False).head(3)


,ventas_unicas,articulos_vendidos,productos_distinitos,total_flete,precio_total
seller_id,,,,,
4869f7a5dfa277a7dca6462dcf3b52b2,1132,1156,95,20168.07,229472.63
53243585a1d6dc2643021fd1853d8905,358,410,23,13080.63,222776.05
4a3ca9315b744ce9f8e9374361493884,1806,1987,399,35067.04,200472.92


In [10]:
order_reviews.head(2)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13


In [11]:
#En reseñas, reemplaza los comentarios nulos por la cadena ” y cuenta cuántos comentarios
#quedan vacíos
order_reviews["review_comment_message"]=order_reviews["review_comment_message"].fillna("nulo")
# no me gusta el vaçio, vamos a cambiarlo por "me da flojerinha hacer un review"
order_reviews["review_comment_message"] = order_reviews["review_comment_message"].str.replace("nulo","me da flojerinha hacer un review")


In [12]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,me da flojerinha hacer un review,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,me da flojerinha hacer un review,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,me da flojerinha hacer un review,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [13]:
# problema 29 En pagos, crea valor_por_cuota dividiendo payment_value entre payment_installments,evitando divisiones entre cero o nulos
payments_29 = payments[payments["payment_installments"]!=0]
payments_29 = payments_29.dropna()
payments_29["valor_por_cuta"] =  payments_29["payment_value"] / payments_29["payment_installments"] 

In [14]:
payments.columns

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')

In [15]:
#49 dentifica los pedidos cuyo total de pagos no coincide con el total de precio más flete de sus
#artículos. Construye ambos totales por order_id antes de compararlos y define una tolerancia
#numérica
df_49 = order_items.merge(payments, on="order_id", how ="left")
df_49 = df_49.groupby(["payment_type"]).agg(
    suma_productos = ("price", "sum"),
    suma_flete = ("freight_value", "sum"),
    suma_payment =("payment_value", "sum")
)
df_49

,suma_productos,suma_flete,suma_payment
payment_type,,,
boleto,2391525.66,450714.63,4059699.60
credit_card,10974357.30,1749557.58,15589028.22
debit_card,183758.74,31548.00,253533.86
voucher,659473.64,125608.30,405873.03


In [16]:
df_49["diferencia"] = df_49["suma_payment"] - df_49["suma_flete"] - df_49["suma_productos"]
df_positivo = df_49[df_49["diferencia"]>0]
df_negativo = df_49[df_49["diferencia"]<0]
df_neutro = df_49[df_49["diferencia"]==0]

In [17]:
df_49

,suma_productos,suma_flete,suma_payment,diferencia
payment_type,,,,
boleto,2391525.66,450714.63,4059699.60,1217459.31
credit_card,10974357.30,1749557.58,15589028.22,2865113.34
debit_card,183758.74,31548.00,253533.86,38227.12
voucher,659473.64,125608.30,405873.03,-379208.91


In [18]:
df_positivo["diferencia"].sum()

np.float64(4120799.77)

In [19]:
df_negativo["diferencia"].sum()

np.float64(-379208.91)

# Rúbrica

## Nivel 1 (1)
* 3
## Nivel 2 (1)
* 3
## Nivel 3 (1.5)
* 5
## Nivel 4 (1.8)
* 5
## Nivel 5 (2)
* 5

no vienen: 40

In [20]:
# problema 25Crea en pedidos una columna duracion_entrega_dias como diferencia entre fecha de entrega
#real y fecha de compra. Calcula la mediana sólo para pedidos con ambas fechas disponibles

# hago mi tabla pa no batallar en los siguientes problemas
df_25 = orders
# convierto las fechas a tipo fecha (datetime)
df_25["order_purchase_timestamp"] = pd.to_datetime(df_25["order_purchase_timestamp"])
df_25["order_delivered_customer_date"] = pd.to_datetime(df_25["order_delivered_customer_date"])
# ya en formato fecha puedo restarlos
df_25["duracion_entrega_dias"] = df_25["order_delivered_customer_date"] - df_25["order_purchase_timestamp"]
df_25.head()

print(df_25["duracion_entrega_dias"].median())
print(df_25["duracion_entrega_dias"].mean())

10 days 05:13:34
12 days 13:24:31.879068369


In [21]:
customers.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')

In [28]:
df_25.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'duracion_entrega_dias'],
      dtype='object')

In [32]:
print(sellers.columns)
print(orders.columns)
print(df_47.columns)

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'duracion_entrega_dias'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'duracion_entrega_dias', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state'],
      dtype='object')


In [27]:
# 47 Calcula el tiempo promedio entre compra y entrega real por estado del cliente. Une pedidos
#con clientes, elimina registros sin fechas válidas y ordena los estados por duració

# aquí voy a reciclar el df del problema 25 pero le falta la información del estado del customer
df_47 = df_25

# a df_47 le pegamos la información del estado que está en customer y lo pegamos con "custemr_id"
df_47 = df_47.merge(customers, on = "customer_id", how = "left")
df_47.groupby("customer_state","seller_state")[["duracion_entrega_dias"]].max().sort_values(by="duracion_entrega_dias",ascending=False)

ValueError: No axis named seller_state for object type DataFrame

In [24]:
order_items.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')

In [25]:
sellers.columns

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')

In [26]:
df1 = sellers.merge(order_items, on="seller_id", how ="left")
df2 = df1.groupby("seller_id").agg(
    suma_precio =("price", "sum"),
    suma_flete = ("freight_value","sum")
)
df2["valor_neto"] = df2["suma_precio"] - df2["suma_flete"]
display(df2.sort_values(by="valor_neto", ascending=False).head(10))

,suma_precio,suma_flete,valor_neto
seller_id,,,
53243585a1d6dc2643021fd1853d8905,222776.05,13080.63,209695.42
4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,20168.07,209304.56
fa1c13f2614d7b5c4749cbc52fecda94,194042.03,10042.70,183999.33
7e93a43ef30c4f03f38b393420bc753a,176431.87,6322.18,170109.69
4a3ca9315b744ce9f8e9374361493884,200472.92,35067.04,165405.88
7c67e1448b00f6e969d365cea6b010ab,187923.89,51612.55,136311.34
da8622b14eb17ae2831f4ac5b9dab84a,160236.57,24955.75,135280.82
7a67c85e85bb2ce8582c35f2203ad736,141745.53,20902.85,120842.68
46dc3b2cc0980fb8ec44634e21d2718e,128111.19,11798.50,116312.69


In [ ]:
class Producto:
    def __init__(self, precio, nombre):
        self.precio = precio
        self.nombre = nombre



3

In [ ]:
carrito[producto3] = cantidad


In [ ]:
carrito = {producto1 : 5, producto2 : 9, producto3: cantidad}
total = 0
for producto in self.carrito:
    producto.vender(self.carrito[producto])

if dinero te alcanza:
    for producto in carrito.keys():
        producto.vender(carrito[producto])
else:
    no te alcanza

[1, 2, 2]

In [ ]:
def __repr__(self):
    return self.nombre 